In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_recall_curve, average_precision_score


np.random.seed(42)

n_samples = 1200

distance_to_fault_km = np.random.uniform(0, 10, n_samples)
alteration_index = np.random.uniform(0, 1, n_samples)
geochem_anomaly = np.random.uniform(0, 1, n_samples)

lithology = np.random.choice(
    ["granite", "schist", "basalt", "gneiss", "sedimentary"],
    size=n_samples
)

fault_influence = np.exp(-distance_to_fault_km)

mineral_potential = (
    (fault_influence > 0.4) &
    (alteration_index > 0.6) &
    (geochem_anomaly > 0.6)
).astype(int)

df = pd.DataFrame({
    "distance_to_fault_km": distance_to_fault_km,
    "fault_influence": fault_influence,
    "alteration_index": alteration_index,
    "geochem_anomaly": geochem_anomaly,
    "lithology": lithology,
    "mineral_potential": mineral_potential
})

df = pd.get_dummies(df, columns=["lithology"], drop_first=True)

X = df.drop("mineral_potential", axis=1)
y = df["mineral_potential"]

model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

all_probs = []
all_true = []

for train, test in cv.split(X, y):
    model.fit(X.iloc[train], y.iloc[train])
    probs = model.predict_proba(X.iloc[test])[:, 1]
    all_probs.extend(probs)
    all_true.extend(y.iloc[test])

precision, recall, _ = precision_recall_curve(all_true, all_probs)
ap_score = average_precision_score(all_true, all_probs)

plt.figure(figsize=(7,5))
plt.plot(recall, precision, linewidth=2)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.tight_layout()
plt.show()

model.fit(X, y)

importances = model.feature_importances_
features = X.columns

plt.figure(figsize=(7,5))
plt.barh(features, importances)
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


x_range = np.linspace(0, 10, 100)
y_range = np.linspace(0, 1, 100)
xx, yy = np.meshgrid(x_range, y_range)

fault_inf_grid = np.exp(-xx)

grid_df = pd.DataFrame({
    "distance_to_fault_km": xx.ravel(),
    "fault_influence": fault_inf_grid.ravel(),
    "alteration_index": yy.ravel(),
    "geochem_anomaly": 0.7
})

for col in X.columns:
    if col.startswith("lithology_"):
        grid_df[col] = 0

# Set target lithology for visualization (example: schist)
grid_df["lithology_schist"] = 1

proba = model.predict_proba(grid_df[X.columns])[:, 1]
proba_map = proba.reshape(xx.shape)

plt.figure(figsize=(7,5))
plt.contourf(xx, yy, proba_map, levels=20, cmap="inferno")
plt.colorbar(label="Prospectivity")
plt.xlabel("Distance to Fault (km)")
plt.ylabel("Alteration Index")
plt.tight_layout()
plt.show()